## 11. HNSW参数调优难？掌握SQ8量化压缩技术实现速度与准确率平衡

在使用 Faiss 的 HNSW（Hierarchical Navigable Small World）索引时，虽然其搜索速度和准确率都很高，但内存占用大 是其主要缺点之一。

为了解决这个问题，Faiss 提供了 SQ8（Scalar Quantization 8 bits）量化压缩技术 ，可以在几乎不损失精度的前提下显著减少内存占用。

下面我们将通过一个完整的示例，展示：

1. 原始 HNSW 索引的构建与搜索；
2. 使用 SQ8 量化后的 HNSW 构建与搜索；
3. 比较两者在 内存占用、搜索速度、召回率 上的表现。

### 1 安装 Faiss

确保你安装的是支持 GPU 和 SQ8 的版本

### Attempting Faiss GPU Installation

Since you've selected an A100 GPU, let's try to install `faiss-gpu` to utilize it. Installing `faiss-gpu` can sometimes be tricky due to specific CUDA version requirements, but we'll try a common approach.

First, we'll uninstall `faiss-cpu` to prevent conflicts, then install `faiss-gpu`, and finally verify if Faiss can see your GPU.

In [5]:
# Uninstall faiss-cpu to avoid conflicts when installing faiss-gpu
print("Uninstalling faiss-cpu if present...")
!pip uninstall faiss-cpu -y
print("faiss-cpu uninstallation complete.")

Uninstalling faiss-cpu if present...
faiss-cpu uninstallation complete.


In [6]:
# Attempt to install faiss-gpu
print("Attempting to install faiss-gpu...")
!pip install faiss-gpu-cu12
print("faiss-gpu installation attempt complete.")

Attempting to install faiss-gpu...
faiss-gpu installation attempt complete.


In [7]:
import faiss

# Verify if Faiss was installed with GPU support
if hasattr(faiss, 'get_num_gpus'):
    num_gpus = faiss.get_num_gpus()
    if num_gpus > 0:
        print(f"Faiss detected {num_gpus} GPU(s). You are now using faiss-gpu!")
    else:
        print("Faiss-gpu installed, but no GPUs detected by Faiss. Falling back to CPU.")
        print("If you still encounter issues, you may need to restart the runtime or use `faiss-cpu`.")
else:
    print("Faiss-gpu does not seem to be correctly installed or detected.")
    print("Please ensure CUDA is properly configured or consider using `faiss-cpu`.")

# If faiss-gpu installation fails again, you might need to try a specific wheel version or revert to faiss-cpu.
# The rest of the notebook cells can now proceed, assuming faiss-gpu is functional or faiss-cpu acts as fallback.

Faiss detected 1 GPU(s). You are now using faiss-gpu!


## 2. 示例代码：HNSW vs HNSW + SQ8 对比

In [8]:
import numpy as np
import time

# Set random seed for reproducibility
np.random.seed(42)

# Construct the dataset
d = 128          # Vector dimension
nb = 100000      # Database size
nq = 1000        # Number of queries
k = 10

xb = np.random.random((nb, d)).astype("float32")
xq = np.random.random((nq, d)).astype("float32")

# Optional: normalize if you want cosine similarity
# faiss.normalize_L2(xb)
# faiss.normalize_L2(xq)


In [9]:
# -----------------------------
# 1. Build original HNSW index
# -----------------------------
index_hnsw = faiss.IndexHNSWFlat(d, 32)
index_hnsw.hnsw.efConstruction = 200
index_hnsw.hnsw.efSearch = 64

In [10]:
print("Start building original HNSW...")
start_time = time.time()
index_hnsw.add(xb)
end_time = time.time()
print(f"Original HNSW build time: {end_time - start_time:.2f}s")

# Query test
D, I = index_hnsw.search(xq, k)
print("Original HNSW top-10 query results (first 5 queries):")
print(I[:5])

# Very rough memory estimate for raw vectors only
print(
    f"Original HNSW raw vector memory (~): "
    f"{index_hnsw.ntotal * d * 4 / (1024 ** 2):.2f} MB"
)

Start building original HNSW...
Original HNSW build time: 37.70s
Original HNSW top-10 query results (first 5 queries):
[[19523 72727 34414 30313  7176 86931  4934 20772 91194 50244]
 [ 5601 69743  1379 16978 75556 94079 36324 52841 71993 66441]
 [31536 28371  7009 27092 23427 30305 12762 50619 74453 35754]
 [12625 72189 23505 65220 29097 42663 55150 20835 39616 68382]
 [97156 51507 21281 41110 88286 32283 61565 78278 23634 93393]]
Original HNSW raw vector memory (~): 48.83 MB


In [11]:
# -----------------------------------
# 2. Build HNSW + SQ8 quantized index
# -----------------------------------
# This is the real HNSW + SQ8 index
index_hnsw_sq8 = faiss.IndexHNSWSQ(
    d,
    faiss.ScalarQuantizer.QT_8bit,
    32
)
index_hnsw_sq8.hnsw.efConstruction = 200
index_hnsw_sq8.hnsw.efSearch = 64

print("Start building HNSW+SQ8...")
start_time = time.time()
index_hnsw_sq8.train(xb)   # train is required for SQ-based index
index_hnsw_sq8.add(xb)
end_time = time.time()
print(f"HNSW+SQ8 build time: {end_time - start_time:.2f}s")

# Query test
D2, I2 = index_hnsw_sq8.search(xq, k)
print("HNSW+SQ8 top-10 query results (first 5 queries):")
print(I2[:5])

# Rough memory estimate: SQ8 uses ~1 byte per dimension
print(
    f"HNSW+SQ8 raw code memory (~): "
    f"{index_hnsw_sq8.ntotal * d * 1 / (1024 ** 2):.2f} MB"
)

Start building HNSW+SQ8...
HNSW+SQ8 build time: 351.93s
HNSW+SQ8 top-10 query results (first 5 queries):
[[19523 72727 34414 30313 31191  7176  4934 86931 20772 93869]
 [ 5601 69743  1379 16978 75556 94079 85888 25843 36324 52841]
 [31536 28371 32815  7009 27092 23427 30305 12762 50619 73324]
 [22844 12625 29550 72189 23505 65220 29097 42663 84819 55150]
 [97156 51507 21281 41110 88286 61565 32283 78278 23634 93393]]
HNSW+SQ8 raw code memory (~): 12.21 MB


In [12]:
# -----------------------------------
# 3. Recall calculation
# -----------------------------------
# Top-1 recall against original HNSW
recall_top1 = np.mean(I[:, 0] == I2[:, 0])

# Top-k overlap recall
recall_at_k = np.mean([
    len(set(I[i]) & set(I2[i])) / k
    for i in range(nq)
])

print(f"Top-1 Recall between HNSW and HNSW+SQ8: {recall_top1 * 100:.2f}%")
print(f"Top-{k} overlap recall: {recall_at_k * 100:.2f}%")

Top-1 Recall between HNSW and HNSW+SQ8: 84.10%
Top-10 overlap recall: 84.83%


## 3 HNSW 参数详解

| 参数名            | 默认值     | 含义说明                                     | 调整建议 |
|------------------|------------|----------------------------------------------|----------|
| `M`              | 32         | 每个节点连接的最大邻居数，控制图的复杂度     | 增大 M 提高召回率，但增加内存和构建时间 |
| `efConstruction` | 200        | 构建索引时的搜索范围，影响图质量             | 更高质量 = 更大的 efConstruction，适合数据集复杂或对精度要求高的场景 |
| `efSearch`       | 10         | 搜索时的探索范围，直接影响准确率与速度     | 增大 efSearch 可提升准确率，但会降低查询速度 |
| `level_mult`     | 1 / log(M) | 控制层级分布密度                             | 一般不调整，除非有特殊结构需求 |

适用场景：
- M：图的连接密度
场景：需要高召回率 → 增大 M；
场景：内存敏感、快速部署 → 减小 M；

- efConstruction：构建阶段的质量控制
场景：离线训练、追求精确 → 增大该值；

- efSearch：查询阶段的探索深度
场景：实时性要求高 → 适当减小以加快响应；

- level_mult：层级分布系数
场景：通常默认即可，仅在自定义图结构时调整。

## 4 SQ8 vs SQ16

SQ8（Scalar Quantization 8-bit）
- 优点：压缩率高，内存最小化；
- 缺点：精度略低；
- 场景：适用于移动端、嵌入式设备、轻量级推荐系统。

SQ16（Scalar Quantization 16-bit）
- 优点：保留更高精度；
- 缺点：内存占用翻倍；
- 场景：适用于云服务、图像检索、视频匹配等对精度要求较高的任务。


## 总结
| 需求类型       | 推荐方案                          | 理由说明                                                                 |
|----------------|-----------------------------------|--------------------------------------------------------------------------|
| **内存敏感**   | HNSW + SQ8                        | 每个向量仅占 1 字节，适合嵌入式/移动端或内存受限环境                       |
| **精度优先**   | HNSW + SQ16 / PQ / OPQ            | 提供更高精度，适合图像检索、视频匹配等对召回率要求高的任务                 |
| **查询速度快** | HNSW + SQ8 + efSearch=20~40       | 在保持合理准确率的同时加快响应速度，适合实时系统                           |
| **大规模部署** | HNSW + SQ8 + GPU 加速             | 支持海量数据快速训练与查询，适用于线上服务                                 |
| **图像特征匹配** | HNSW + SQ8 或 SQ16                | 对于 128D 等常见维度，SQ8 已足够；若需更高精度可用 SQ16                     |
| **视频检索系统** | HNSW + SQ16                       | 视频特征通常更复杂，需要更高精度以提升匹配准确性                           |
| **推荐系统应用** | HNSW + SQ8                        | 在保证性能的前提下节省内存，适合用户/物品 Embedding 的高效匹配              |
| **压缩与精度平衡** | HNSW + PQ / OPQ                  | 使用乘积量化技术，在压缩模型大小的同时保持较高搜索准确率                   |